# Practical: Structured Output Validation (Pydantic)

In production, raw text output is useless for automation. We need structured data (JSON) and we need to **validate** that it conforms to our schema.

This notebook demonstrates the industry-standard way to get validated JSON using **Pydantic** and OpenAI's native **Structured Outputs**.

In [ ]:
# !pip install pydantic openai

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional
from openai import OpenAI
import json

class Entity(BaseModel):
    name: str = Field(description="The name of the person or organization")
    role: str = Field(description="Their role or relationship")
    sentiment: str = Field(description="Positive, Negative, or Neutral")

class ExtractionResult(BaseModel):
    entities: List[Entity]
    summary: str
    confidence_score: float = Field(ge=0, le=1)

# 1. Setup Client
client = OpenAI(api_key="sk-...")

def extract_info(text):
    # OpenAI native pydantic integration
    completion = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Extract structured information from the text."},
            {"role": "user", "content": text}
        ],
        response_format=ExtractionResult,
    )

    return completion.choices[0].message.parsed

text = "Apple CEO Tim Cook announced a new AI initiative yesterday. The market reacted positively, although some critics remain skeptical."
# result = extract_info(text)
# print(result.model_dump_json(indent=2))
print("Code ready. Pydantic schema defined and parser integrated.")

## Why this matters in production:

1. **Type Safety**: Your downstream code can access `result.entities[0].name` without worrying about dictionary key errors.
2. **Schema Enforcement**: The model is mathematically constrained (via the API) to follow the Pydantic schema.
3. **Self-Correction**: If you don't use the `.parse()` method, you can catch `ValidationError` and prompt the model to fix its own JSON.

## 2. Advanced: Native 'Structured Output' vs Manual Catching

Modern APIs (OpenAI `gpt-4o`, `gpt-4o-mini`) support **Strict Mode**. This uses constrained decoding (CFG) to *guarantee* the model follows your schema.

```python
# The BEST way (Guaranteed Schema)
def get_guaranteed_json(text):
    completion = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[...],
        response_format=ExtractionResult,
    )
    return completion.choices[0].message.parsed

# The OLD/FALLBACK way (Manual Validation)
def get_manual_json(text):
    response = client.chat.completions.create(
        model="other-model",
        messages=[...],
        response_format={"type": "json_object"}
    )
    raw_json = json.loads(response.choices[0].message.content)
    # Manual validation with Pydantic
    return ExtractionResult(**raw_json)
```